## Imports & Startup

In [ ]:
import gc
import glob
import json
import math
import os
import pathlib
import queue
import re
import signal
import subprocess
import sys
import tempfile
import threading
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from typing import Optional, TypedDict

import pandas as pd
import polars as pl
import torch
from jupyter_client import KernelManager
from openai import OpenAI
from transformers import AutoTokenizer, set_seed

import kaggle_evaluation.aimo_3_inference_server

# ── Startup ────────────────────────────────────────────────────────────────────
overall_start_time = time.time()
print(datetime.now())

LOKAL = True
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TIKTOKEN_ENCODINGS_BASE'] = './tmp/tiktoken_encodings'

LEAN_TOOL = True
ATTEMPTS = 6 if not LOKAL else 1
WARMUP_TIMEOUT = 50  # seconds — Mathlib import takes a while on first load

## SolverConfig

In [ ]:
class SolverConfig:
    system_prompt = (
        'You are an elite mathematical problem solver with expertise at the International '
        'Mathematical Olympiad (IMO) level. Your goal is to find the correct answer through '
        'rigorous mathematical reasoning.\n\n'

        '# Problem-Solving Approach:\n'
        '1. UNDERSTAND: Carefully read and rephrase the problem in your own words. '
        'Identify what is given, what needs to be found, and any constraints.\n'
        '2. EXPLORE: Consider multiple solution strategies. Think about relevant theorems, '
        'techniques, patterns, or analogous problems. Don\'t commit to one approach immediately.\n'
        '3. PLAN: Select the most promising approach and outline key steps before executing.\n'
        '4. EXECUTE: Work through your solution methodically. Show all reasoning steps clearly.\n'
        '5. TOOL_USE: Use your tools PYTHON_EXECUTION and LEAN_PROOF to validate your reasoning.\n'
        '6. VERIFY: Check your answer by substituting back, testing edge cases, or using '
        'alternative methods. Ensure logical consistency throughout.\n\n'

        '# Mathematical Reasoning Principles:\n'
        '- Break complex problems into smaller, manageable sub-problems\n'
        '- Look for patterns, symmetries, equivalences, equalities, and special cases that provide insight\n'
        '- Use concrete examples to build intuition before generalizing\n'
        '- Consider extreme cases and boundary conditions\n'
        '- If stuck, try working backwards from the desired result\n'
        '- Be willing to restart with a different approach if needed\n'
        '- When Python reveals a pattern over finite cases, ask: does this hold universally? '
        'If so, use lean_repl to prove it — a formal proof eliminates all doubt and is often '
        'the decisive step for problems involving divisibility, congruences, or algebraic identities.\n\n'

        '# Verification Requirements:\n'
        '- Cross-check arithmetic and algebraic manipulations\n'
        '- Verify that your solution satisfies all problem constraints\n'
        '- Test your answer with simple cases or special values when possible\n'
        '- Ensure dimensional consistency and reasonableness of the result\n'
        '- If your answer depends on a claim of the form "for all n ..." or "for all primes ...", '
        'prove it with lean_repl rather than trusting finite evidence alone.\n\n'

        '# Output Format:\n'
        'The final answer must be a non-negative integer between 0 and 99999.\n'
        'Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}\n\n'

        'Think step-by-step and show your complete reasoning process. Quality of reasoning '
        'is as important as the final answer.'
    )

    python_tool_prompt = (
        'CORRECT USE — call Python for:\n'
        '- Computing specific values: counts, sums, gcd, modular results, factorizations\n'
        '- Pattern search: generate examples, detect regularities, test conjectures on small cases\n'
        '- Solving equations or optimizing over a concrete domain\n'
        '- Any arithmetic that would take more than 2 sentences to write by hand\n\n'

        'WRONG USE — do NOT call Python for:\n'
        '- Proving something holds for ALL n (use lean_repl for universal statements)\n'
        '- Writing code that just restates the problem without computing anything\n\n'

        'Environment: stateful Jupyter notebook — variables persist between calls. '
        'Always use print() for results. Available: numpy, sympy, itertools, collections.'
    )

    lean_tool_prompt = (
        'Use this tool at the maximum of ONCE to formally prove or disprove a UNIVERSAL mathematical statement using Lean 4.\n\n'

        'WHY USE LEAN:\n'
        'Many competition problems hinge on a universal lemma — a claim that holds for all n, '
        'all primes, or all elements of an infinite set. Python can only check finite evidence; '
        'Lean gives you a rigorous proof. A proven lemma can be the decisive step that unlocks '
        'the final answer with certainty. If you notice a pattern in Python output that seems to '
        'hold universally, prove it with Lean before relying on it.\n\n'

        'CORRECT USE — statements that hold for infinitely many cases:\n'
        '- Algebraic identities: "for all integers n, ..." or "for all primes p, ..."\n'
        '- Divisibility or modular arithmetic relationships over an infinite domain\n'
        '- Inequalities or equivalences that Python cannot verify exhaustively\n'
        '- Proving that a certain structure (symmetry, periodicity, congruence) holds universally\n\n'

        'WRONG USE — do NOT call this tool for:\n'
        '- Computing a specific number or checking a single case (use Python instead)\n'
        '- Verifying that f(42) = 17 or that gcd(a, b) = 1 for given a, b (use Python)\n'
        '- Speeding up a slow Python calculation (Lean does not compute faster)\n'
        '- Any question of the form "what is the value of X?" (Python handles values)\n\n'

        'Decision rule: if your statement contains a specific numeric value as the main subject '
        '(not as a bound or parameter), use Python. '
        'If your statement starts with "for all" or "there exists" over an infinite set, use Lean.\n\n'

        'HOW TO USE:\n'
        'Provide informal_statement_content (natural language, e.g. "For all n ≥ 1, n² + n is even") '
        'and a theorem_name.\n'
        'Returns: proven / failed / technical error.\n\n'

        'A proven result gives a rigorous guarantee for infinitely many cases. '
        'A failure likely means the statement is wrong — reconsider your approach.'
    )

    preference_python_prompt = (
        'CORRECT USE of Python — call it for:\n'
        '- Any specific numerical value, count, sum, gcd, factorization, modular result\n'
        '- Pattern search: compute examples, look for regularities, test conjectures on small cases\n'
        '- Equation solving, optimization, root-finding over concrete domains\n'
        '- Checking whether a formula or approach works before committing to it\n\n'

        'WRONG USE — do NOT use Python for:\n'
        '- Proving something holds for ALL integers/primes/etc. (use lean_repl for universal proofs)\n'
        '- Restating or summarizing the problem in code comments instead of computing\n\n'

        'Available libraries and their purpose:\n'
        '- sympy: exact symbolic algebra, number theory (primes, divisors, modular arithmetic), '
        'equation solving, polynomial factorization — use for exact answers\n'
        '- numpy: large-scale numerical computation, linear algebra, matrix operations\n'
        '- itertools / collections: combinatorics, permutations, counting, grouping\n\n'

        'Strategy:\n'
        '- Prefer sympy over manual arithmetic for exact results\n'
        '- If brute-force is too slow, derive a smarter algorithm or closed-form formula first\n'
        '- Verify numerical results against known special cases or theoretical bounds'
    )

    served_model_name = 'qwen3.5-27b'
    model_path = './data/llm_data' if LOKAL else '/kaggle/input/models/shelterw/qwen3.5/transformers/qwen3.5-27b-fp8/1'
    get_embeddings = True
    embedding_path = "./data/llm_embedding/embeddings.pt" if LOKAL else '/kaggle/input/datasets/frederikbeats/qwen3-5-27b-fp8-embeddingdata/embeddings.pt'
    kv_cache_dtype = 'fp8_e4m3'
    dtype = 'auto'
    quantization = ''
    spec_config = {
        "method": "mtp",
        "num_speculative_tokens": 3
    }
    compilation_config = {
        "cudagraph_capture_sizes": [1, 2, 4, 8, 16, 24, 32]
    }
    load_format_flag = quantization

    re_pattern = r"<parameter=[^>]+>(.*?)</parameter>"

    high_problem_timeout = 500
    base_problem_timeout = 250

    notebook_limit = 17400
    server_timeout = 500

    session_timeout = 18000 # That at not LOKAL the client is always on while 5h
    jupyter_timeout = 15 # Jeder Python Execution
    sandbox_timeout = 3

    stream_interval = 200
    context_tokens = 960000 if not LOKAL else 16384
    max_new_tokens = 8192  # Cap pro Completion — vLLM reserviert sonst bis zu 64K KV-Slots → OOM
    guessing_tokens = 128
    buffer_tokens = guessing_tokens # Damit noch was fürs Guessing übrig bleibt
    search_for_answer_tokens = 32
    top_logprobs = 20
    top_logprobs_search_for_wait = 4
    batch_size = max(compilation_config["cudagraph_capture_sizes"]) # int(attempts//4 *3) Crashed leider, Vielleicht wurde es mit 16 trainiert
    early_stop = max(1,int(ATTEMPTS//2))
    workers = 16
    turns = 156
    seed = 42
    profer_cuncurrent = 1 if LOKAL else 1

    if LEAN_TOOL:
        if LOKAL:
            gpu_memory_utilization = 0.76
        else:
            gpu_memory_utilization = 0.66  # H100 80GB: 66% × 80 = 53GB — gibt Form/Prov je 15%, ~3GB Puffer
    else:
        gpu_memory_utilization = 0.95
    frequency_enable_thinking = 3

    temperature = 1.0
    # min_p = 0.02
    presence_penalty = 1.5
    port = 8000

    tools = [
        {
            "type": "function",
            "function": {
                "name": "python",
                "description": python_tool_prompt,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "code": {
                            "type": "string",
                            "description": "The Python code to execute."
                        }
                    },
                    "required": ["code"]
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "lean_repl",
                "description": lean_tool_prompt,
                "parameters": {
                    "type": "object",
                    "properties": {
                        "informal_statement_content": {
                            "type": "string",
                            "description": (
                                "The mathematical statement to prove or validate, "
                                "in natural language or semi-formal notation. "
                                "Include also all pre-conditions for the input symbols"
                                "and post-conditions for the output symbols. Use Latex Notations."
                                "Example: 'For all natural numbers n, n + 0 = n'"
                            )
                        },
                        "theorem_name": {
                            "type": "string",
                            "description": (
                                "A unique describing name for the theorem"
                            )
                        },
                    },
                    "required": ["informal_statement_content", "theorem_name"]
                }
            }
        }
    ]
    if not LEAN_TOOL:
        tools = [tools[0]]

set_seed(SolverConfig.seed)

## FormConfig

In [ ]:
class FormConfig:
    system_prompt = (
        'You are an elite Lean Code formalizer.\n\n'
    )

    user_prompt = ''

    lean_code_tool_prompt = (
        'Use this tool to formalize a natural language problem statement including context and name into valid Lean 4 code.'
    )

    served_model_name = 'form'
    model_path = '/home/frederik/.cache/huggingface/hub/models--Goedel-LM--Goedel-Formalizer-V2-8B/snapshots/fe2d362d899601abe79d7d5e95eaa7fe9883a0cb' if LOKAL else '/kaggle/input/models/frederikbeats/goedel-formalizer-v2-8b/transformers/default/1'
    get_embeddings = False

    kv_cache_dtype = 'fp8'if not LOKAL else 'auto'
    dtype = 'auto'
    quantization = 'fp8' if not LOKAL else 'bitsandbytes'
    spec_config = {}
    speculative_flag = ''
    compilation_config = {}
    load_format_flag = '' if not LOKAL else 'bitsandbytes'
    max_new_tokens = 1024

    re_pattern = r"<parameter=[^>]+>(.*?)</parameter>"

    server_timeout = 500

    session_timeout = 18000 # That at not LOKAL the client is always on while 5h

    context_tokens = 4096 # Laut Huggingface "max_position_embeddings": 40960,
    buffer_tokens = 512

    batch_size = 8
    workers = 16
    turns = 3
    seed = 42

    if LEAN_TOOL:
        if LOKAL:
            gpu_memory_utilization = 0.18
        else:
            gpu_memory_utilization = 0.14
    else:
        gpu_memory_utilization = 0.00

    temperature = 0.0
    port = 8001

set_seed(FormConfig.seed)

## ProverConfig

In [ ]:
class ProverConfig:
    system_prompt = (
        'Complete the following Lean 4 code:\n\n'

        '```lean4\n{formal_statement}```\n\n'

        'Before producing the Lean 4 code to formally prove the given theorem, provide a detailed proof plan outlining the main proof steps and strategies.\n'
        'The plan should highlight key ideas, intermediate lemmas, and proof structures that will guide the construction of the final formal proof.\n\n'
    )

    served_model_name = 'prov'
    model_path = '/home/frederik/.cache/huggingface/hub/models--Goedel-LM--Goedel-Code-Prover-8B/snapshots/858f3b5e04bf24aca3a1a113ea1889a4de9ed4ef' if LOKAL else '/kaggle/input/models/frederikbeats/goedel-prover-v2-8b/transformers/default/1'
    get_embeddings = False

    kv_cache_dtype = 'fp8' if not LOKAL else 'auto'
    dtype = 'auto'
    quantization = 'fp8' if not LOKAL else 'bitsandbytes'
    spec_config = {}
    speculative_flag = ''
    compilation_config = {}
    load_format_flag = '' if not LOKAL else 'bitsandbytes'

    re_pattern = r"<parameter=[^>]+>(.*?)</parameter>"

    server_timeout = 500
    proof_timeout = 300 if not LOKAL else 1000

    session_timeout = 18000  # That at not LOKAL the client is always on while 5h

    prover_llm_timeout = 200 if not LOKAL else 1000
    max_new_tokens = 20000  # Cap pro Completion — vLLM reserviert sonst ~39k KV-Slots → Preemption
    ProofCorrection_Rounds = 3
    context_tokens = 32960 if not LOKAL else 16384  # Laut Huggingface "max_position_embeddings": 40960, # Max laut src/inferene: 131072
    buffer_tokens = 512
    attempts = SolverConfig.profer_cuncurrent  # concurrent Prover.proof()-Calls (= SolverConfig.profer_cuncurrent not LOKAL)
    variants_per_round = 2  # LLM-Varianten pro Korrektur-Runde
    batch_size = 8  # max gleichzeitige LLM-Requests an den Prover-Server
    workers = 16

    seed = 42

    if LEAN_TOOL:
        if LOKAL:
            gpu_memory_utilization = 0.22
        else:
            gpu_memory_utilization = 0.18
    else:
        gpu_memory_utilization = 0.00

    temperature = 1.0
    port = 8002

set_seed(ProverConfig.seed)

## ChatTemplate

In [ ]:
class ChatTemplate:

    def apply_chat_template_for_chatml_for_Solver(self, messages, tokenizer) -> list:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=True,
            tools=SolverConfig.tools,
        )

    def apply_chat_template_for_chatml_for_Solver_final(self, messages, tokenizer) -> list:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=False,
        )

    def apply_chat_template_for_chatml_for_Form(self, messages, tokenizer) -> list:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            enable_thinking=True,
        )

    def apply_chat_template_for_chatml_for_Prov(self, messages, tokenizer) -> list:
        return self.apply_chat_template_for_chatml_for_Form(messages, tokenizer)

## PythonSandbox

In [ ]:
class PythonSandbox:
    _port_lock = threading.Lock()
    _next_port = 50000

    @classmethod
    def _get_next_ports(cls, count: int = 5) -> list[int]:

        with cls._port_lock:
            ports = list(range(cls._next_port, cls._next_port + count))
            cls._next_port += count

            return ports

    def __init__(self, timeout: float):

        self._default_timeout = timeout
        self._owns_kernel = False
        self._client = None
        self._km = None

        # ports = self._get_next_ports(5)

        env = os.environ.copy()
        env['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'
        env['PYDEVD_WARN_EVALUATION_TIMEOUT'] = '0'
        env['JUPYTER_PLATFORM_DIRS'] = '1'
        env['PYTHONWARNINGS'] = 'ignore'
        env['MPLBACKEND'] = 'Agg'

        self._km = KernelManager()

        self._km.start_kernel(env=env, extra_arguments=['--Application.log_level=CRITICAL'])

        self._client = self._km.blocking_client()
        self._client.start_channels()
        self._client.wait_for_ready(timeout=self._default_timeout)
        self._owns_kernel = True

        self.execute(
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def _format_error(self, traceback: list[str]) -> str:

        clean_lines = []

        for frame in traceback:
            clean_frame = re.sub(r'\x1b\[[0-9;]*m', '', frame)

            if 'File "' in clean_frame and 'ipython-input' not in clean_frame:
                continue

            clean_lines.append(clean_frame)

        return ''.join(clean_lines)

    def execute(self, code: str, timeout: float | None = None) -> str:

        client = self._client
        effective_timeout = timeout or self._default_timeout

        msg_id = client.execute(
            code,
            store_history=True,
            allow_stdin=False,
            stop_on_error=False
        )

        stdout_parts = []
        stderr_parts = []

        start_time = time.time()

        while True:
            elapsed = time.time() - start_time

            if elapsed > effective_timeout:
                self._km.interrupt_kernel()

                return f'[ERROR] Execution timed out after {effective_timeout} seconds'

            try:
                msg = client.get_iopub_msg(timeout=1.0)

            except queue.Empty:
                continue

            if msg.get('parent_header', {}).get('msg_id') != msg_id:
                continue

            msg_type = msg.get('msg_type')
            content = msg.get('content', {})

            if msg_type == 'stream':
                text = content.get('text', '')

                if content.get('name') == 'stdout':
                    stdout_parts.append(text)

                else:
                    stderr_parts.append(text)

            elif msg_type == 'error':
                traceback_list = content.get('traceback', [])

                stderr_parts.append(self._format_error(traceback_list))

            elif msg_type in {'execute_result', 'display_data'}:
                data = content.get('data', {})
                text = data.get('text/plain')

                if text:
                    stdout_parts.append(text if text.endswith('\n') else f'{text}\n')

            elif msg_type == 'status':
                if content.get('execution_state') == 'idle':
                    break

        stdout = ''.join(stdout_parts)
        stderr = ''.join(stderr_parts)

        if stderr:
            return f'{stdout.rstrip()}\n{stderr}' if stdout else stderr

        return stdout if stdout.strip() else '[WARN] No output. Use print() to see results.'

    def close(self):

        try:
            if self._client:
                self._client.stop_channels()
        except Exception:
            pass

        if self._owns_kernel and self._km is not None:
            try:
                self._km.shutdown_kernel(now=True)
            except Exception:
                pass

            try:
                self._km.cleanup_resources()
            except Exception:
                pass


    def reset(self):

        self.execute(
            '%reset -f\n'
            'import math\n'
            'import numpy\n'
            'import sympy\n'
            'import itertools\n'
            'import collections\n'
            'import mpmath\n'
            'mpmath.mp.dps = 64\n'
        )

    def __del__(self):

        self.close()

## Lean — Pfadkonstanten

In [ ]:
if LOKAL:
    DATASET_DIR = pathlib.Path(__file__).parent / 'data/lean_data'
    ELAN_HOME = pathlib.Path(__file__).parent / 'lean_compiler/.elan'
    LEAN_PROJECT_DIR = pathlib.Path(__file__).parent / 'lean_compiler/lean-project'
else:
    DATASET_DIR = pathlib.Path('/kaggle/input/datasets/frederikbeats/lean-mathlib-toolchain')
    ELAN_HOME = pathlib.Path('/root/.elan')
    LEAN_PROJECT_DIR = pathlib.Path('/root/lean-project')

_LEAN_TOOLCHAIN = 'leanprover--lean4---v4.29.0-rc6'
LEAN_BIN = ELAN_HOME / 'toolchains' / _LEAN_TOOLCHAIN / 'bin' / 'lean'
LAKE_BIN = ELAN_HOME / 'toolchains' / _LEAN_TOOLCHAIN / 'bin' / 'lake'

if LOKAL:
    REPL_BIN      = pathlib.Path(__file__).parent / 'tmp/repl-build/.lake/build/bin/repl'
    REPL_BIN_DATA = None
else:
    REPL_BIN      = pathlib.Path('/root/lean-repl/repl')   # wird von setup_lean() extrahiert
    REPL_BIN_DATA = pathlib.Path('/kaggle/input/datasets/frederikbeats/leanrepl/lean-repl.bin')

REPL_POOL_SIZE = ATTEMPTS * 2   # ATTEMPTS × variants_per_round — 16 Kaggle, 2 lokal

## LeanREPLProcess

In [ ]:
class LeanREPLProcess:
    """Persistenter lean-repl Prozess.
    Lädt Mathlib einmalig via warmup() (~9s), danach ~1-2s pro check() statt ~9s.
    Nicht thread-safe — jeder Thread nutzt seinen eigenen Prozess aus dem Pool."""

    _WARMUP_CMD = (
        "import Mathlib\nimport Aesop\n"
        "set_option maxHeartbeats 0\n"
        "open BigOperators Real Nat Topology Rat"
    )

    def __init__(self, repl_bin: pathlib.Path = REPL_BIN,
                 project_dir: pathlib.Path = LEAN_PROJECT_DIR):
        self.repl_bin    = repl_bin
        self.project_dir = project_dir
        self._proc: subprocess.Popen | None = None
        self._base_env: int | None = None
        self._stderr_log: str | None = None
        self.alive = False
        # Queue für stdout-Zeilen — Background-Thread liest kontinuierlich,
        # damit TextIOWrapper-Buffering den select()-Mechanismus nicht blockiert.
        self._output_queue: queue.Queue = queue.Queue()
        self._reader_thread: threading.Thread | None = None

    @staticmethod
    def _get_lake_env(project_dir: pathlib.Path) -> dict:
        """Holt die Umgebungsvariablen, die `lake env` setzen würde.
        Wir rufen `lake env env` auf (das System-`env`-Kommando gibt alle Vars aus)
        und parsen den Output — damit können wir die REPL direkt starten."""
        base_env = os.environ.copy()
        base_env['ELAN_HOME'] = str(ELAN_HOME)
        print('[LeanREPLProcess] Hole lake-Umgebungsvariablen (lake env env) …', flush=True)
        try:
            result = subprocess.run(
                [str(LAKE_BIN), 'env', 'env'],
                cwd=str(project_dir),
                capture_output=True, text=True,
                timeout=120,
                env=base_env,
            )
        except subprocess.TimeoutExpired:
            print('[LeanREPLProcess] lake env env — Timeout nach 120s!', flush=True)
            return base_env
        except Exception as exc:
            print(f'[LeanREPLProcess] lake env env fehlgeschlagen: {exc}', flush=True)
            return base_env
        if result.returncode != 0:
            print(f'[LeanREPLProcess] lake env env rc={result.returncode}: {result.stderr[:300]}', flush=True)
            return base_env
        lake_env = {}
        for line in result.stdout.splitlines():
            if '=' in line:
                k, v = line.split('=', 1)
                lake_env[k] = v
        print(f'[LeanREPLProcess] lake env: LEAN_PATH={lake_env.get("LEAN_PATH", "(nicht gesetzt)")}', flush=True)
        return lake_env

    def start(self, lake_env: dict | None = None) -> bool:
        env = lake_env if lake_env is not None else os.environ.copy()
        env.setdefault('ELAN_HOME', str(ELAN_HOME))
        try:
            fd, self._stderr_log = tempfile.mkstemp(prefix='lean_repl_', suffix='.stderr')
            stderr_fh = os.fdopen(fd, 'w')
            self._proc = subprocess.Popen(
                [str(self.repl_bin)],
                cwd=str(self.project_dir),
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=stderr_fh,
                text=True,
                bufsize=1,
                env=env,
            )
            stderr_fh.close()
            self.alive = True
            self._start_reader_thread()
            return True
        except Exception as exc:
            print(f'[LeanREPLProcess] start failed: {exc}', flush=True)
            return False

    def _start_reader_thread(self) -> None:
        """Startet einen Daemon-Thread, der stdout kontinuierlich in die Queue liest.
        Nötig weil TextIOWrapper die Daten in seinen internen Buffer puffert, sodass
        select() auf dem Raw-fd nichts sieht, obwohl Daten vorhanden sind."""
        def _read():
            try:
                for line in self._proc.stdout:
                    self._output_queue.put(('line', line))
            except Exception:
                pass
            self._output_queue.put(('eof', ''))
        self._reader_thread = threading.Thread(target=_read, daemon=True, name=f'lean-repl-reader-{self._proc.pid}')
        self._reader_thread.start()

    def _print_stderr_tail(self, n: int = 20) -> None:
        """Gibt die letzten n Zeilen der stderr-Logdatei aus."""
        if not self._stderr_log:
            return
        try:
            with open(self._stderr_log) as f:
                lines = f.readlines()
            if lines:
                tail = lines[-n:]
                print(f'[LeanREPLProcess] stderr ({self._stderr_log}, letzte {len(tail)} Zeilen):', flush=True)
                for l in tail:
                    print('  ' + l.rstrip(), flush=True)
            else:
                print(f'[LeanREPLProcess] stderr leer ({self._stderr_log})', flush=True)
        except Exception as e:
            print(f'[LeanREPLProcess] Konnte stderr-Log nicht lesen: {e}', flush=True)

    def warmup(self, timeout: int = WARMUP_TIMEOUT) -> bool:
        """Lädt Mathlib — setzt _base_env. Danach ist check() schnell."""
        print(f'[LeanREPLProcess] Warmup gestartet (timeout={timeout}s, pid={self._proc.pid if self._proc else "?"})', flush=True)
        t0 = time.time()
        resp = self._send({'cmd': self._WARMUP_CMD}, timeout=timeout)
        if resp is None:
            print(f'[LeanREPLProcess] Warmup fehlgeschlagen nach {time.time() - t0:.1f}s', flush=True)
            return False
        print(f'[LeanREPLProcess] Warmup erfolgreich nach {time.time() - t0:.1f}s', flush=True)
        errs = [m for m in resp.get('messages', []) if m.get('severity') == 'error']
        if errs:
            print(f'[LeanREPLProcess] warmup errors: {errs}', flush=True)
            self.alive = False
            return False
        self._base_env = resp.get('env')
        return self._base_env is not None

    def check(self, code: str, stop_event: threading.Event, timeout: int) -> dict:
        if not self.alive or self._base_env is None:
            return {'pass': False, 'complete': False, 'errors': [], 'warnings': [],
                    'system_errors': 'REPL not ready'}
        clean = '\n'.join(l for l in code.splitlines() if not l.strip().startswith('import'))
        if not clean.strip():
            return {'pass': False, 'complete': False, 'errors': [], 'warnings': [],
                    'system_errors': 'empty code after stripping imports'}
        resp = self._send({'cmd': clean, 'env': self._base_env},
                          timeout=timeout, stop_event=stop_event)
        if resp is None:
            return {'pass': False, 'complete': False, 'errors': [], 'warnings': [],
                    'system_errors': 'REPL timeout or process died'}
        return self._parse(resp)

    def _send(self, request: dict, timeout: int,
              stop_event: threading.Event | None = None) -> dict | None:
        if not self.alive or self._proc is None:
            return None
        try:
            self._proc.stdin.write(json.dumps(request) + '\n\n')
            self._proc.stdin.flush()
        except OSError:
            self.alive = False
            return None

        t_start = time.time()
        deadline = t_start + timeout
        last_log = t_start
        accumulated: list[str] = []

        while True:
            if stop_event is not None and stop_event.is_set():
                self._kill()
                return None
            remaining = deadline - time.time()
            if remaining <= 0:
                elapsed = time.time() - t_start
                print(f'[LeanREPLProcess] TIMEOUT nach {elapsed:.1f}s (limit={timeout}s) — Prozess wird gekillt.', flush=True)
                self._print_stderr_tail()
                self._kill()
                return None
            now = time.time()
            if now - last_log >= 10:
                pid = self._proc.pid
                state_info = self._proc_state(pid)
                print(f'[LeanREPLProcess] Warte auf Antwort… ({now - t_start:.0f}s vergangen, noch {remaining:.0f}s) | {state_info}', flush=True)
                last_log = now

            try:
                event, line = self._output_queue.get(timeout=min(0.2, remaining))
            except queue.Empty:
                continue

            if event == 'eof':
                print(f'[LeanREPLProcess] EOF nach {time.time() - t_start:.1f}s — Prozess ist gestorben.', flush=True)
                self._print_stderr_tail()
                self.alive = False
                return None

            accumulated.append(line)
            text = ''.join(accumulated).strip()
            if not text:
                continue
            try:
                return json.loads(text)
            except json.JSONDecodeError:
                continue

    def _kill(self):
        self.alive = False
        if self._proc is not None:
            try:
                self._proc.kill()
                self._proc.communicate(timeout=2)
            except Exception:
                pass
        while True:
            try:
                self._output_queue.get_nowait()
            except queue.Empty:
                break

    @staticmethod
    def _proc_state(pid: int) -> str:
        """Liest /proc/PID/status, wchan und stdin-fd für Diagnose."""
        parts = []
        try:
            with open(f'/proc/{pid}/status') as f:
                info = dict(line.split(':\t', 1) for line in f if ':\t' in line)
            state = info.get('State', '?').strip()
            vmrss = info.get('VmRSS', '?').strip()
            parts.append(f'pid={pid} state={state} mem={vmrss}')
        except Exception:
            parts.append(f'pid={pid} (gestorben?)')
            return ' | '.join(parts)
        try:
            with open(f'/proc/{pid}/wchan') as f:
                parts.append(f'wchan={f.read().strip()}')
        except Exception:
            pass
        try:
            stdin_target = os.readlink(f'/proc/{pid}/fd/0')
            parts.append(f'stdin→{stdin_target}')
        except Exception:
            pass
        try:
            children = []
            for task in glob.glob(f'/proc/{pid}/task/*/children'):
                with open(task) as f:
                    children.extend(f.read().split())
            if children:
                child_info = []
                for cpid in children[:4]:
                    try:
                        with open(f'/proc/{cpid}/status') as f:
                            ci = dict(l.split(':\t', 1) for l in f if ':\t' in l)
                        cstate = ci.get('State', '?').strip()[:1]
                        with open(f'/proc/{cpid}/wchan') as f:
                            cwchan = f.read().strip()
                        cstdin = os.readlink(f'/proc/{cpid}/fd/0')
                        child_info.append(f'{cpid}(st={cstate},wchan={cwchan},stdin→{cstdin})')
                    except Exception:
                        child_info.append(f'{cpid}(?)')
                parts.append('children=' + ' '.join(child_info))
        except Exception:
            pass
        return ' | '.join(parts)

    def _parse(self, resp: dict) -> dict:
        messages = resp.get('messages', [])
        errors   = [m for m in messages if m.get('severity') == 'error']
        warnings = [m for m in messages if m.get('severity') == 'warning']
        passed   = not errors
        complete = passed and not any(
            "declaration uses `sorry`" in w.get('data', '') or
            'failed' in w.get('data', '')
            for w in warnings
        )
        return {'pass': passed, 'complete': complete,
                'errors': errors, 'warnings': warnings, 'system_errors': None}

    def close(self):
        self._kill()

## LeanREPLPool

In [ ]:
class LeanREPLPool:
    """Pool von persistenten LeanREPLProcess-Instanzen.
    Bietet dieselbe check_single()-Schnittstelle wie früher LeanREPL,
    aber mit ~1-2s pro Check statt ~9s."""

    def __init__(self, pool_size: int = REPL_POOL_SIZE):
        self._pool      = queue.Queue()
        self._pool_size = pool_size
        self._lake_env: dict | None = None

    def lean_repl_warmup(self, warmup_timeout: int = WARMUP_TIMEOUT) -> int:
        """Startet alle REPL-Prozesse parallel und warmt sie auf.
        Gibt die Anzahl erfolgreich gestarteter Prozesse zurück."""
        print(f'[LeanREPLPool] Starte {self._pool_size} REPL-Prozesse ...', flush=True)
        t0 = time.time()

        self._lake_env = LeanREPLProcess._get_lake_env(LEAN_PROJECT_DIR)
        lake_env = self._lake_env

        procs = []
        for _ in range(self._pool_size):
            p = LeanREPLProcess()
            if p.start(lake_env=lake_env):
                procs.append(p)
            else:
                print('[LeanREPLPool] Ein Prozess konnte nicht gestartet werden.', flush=True)

        def _warmup_one(p: LeanREPLProcess):
            ok = p.warmup(timeout=warmup_timeout)
            return p, ok

        ready = 0
        with ThreadPoolExecutor(max_workers=len(procs)) as exe:
            for p, ok in exe.map(_warmup_one, procs):
                if ok:
                    self._pool.put(p)
                    ready += 1
                else:
                    p.close()
                    print('[LeanREPLPool] Warmup fehlgeschlagen — Prozess verworfen.', flush=True)

        elapsed = time.time() - t0
        print(f'[LeanREPLPool] {ready}/{self._pool_size} Prozesse bereit ({elapsed:.1f}s)', flush=True)
        return ready

    def check_single(self, code: str, stop_event: threading.Event, timeout: int) -> dict:
        try:
            proc = self._pool.get(timeout=15)
        except queue.Empty:
            return {'pass': False, 'complete': False, 'errors': [], 'warnings': [],
                    'system_errors': 'REPL pool exhausted'}
        try:
            result = proc.check(code, stop_event, timeout)
        except Exception as exc:
            result = {'pass': False, 'complete': False, 'errors': [], 'warnings': [],
                      'system_errors': str(exc)}
            proc.alive = False

        if proc.alive:
            self._pool.put(proc)
        else:
            lake_env = self._lake_env
            def _replace():
                new_p = LeanREPLProcess()
                if new_p.start(lake_env=lake_env) and new_p.warmup(timeout=WARMUP_TIMEOUT):
                    self._pool.put(new_p)
                else:
                    new_p.close()
                    print('[LeanREPLPool] Ersatzprozess konnte nicht gestartet werden.', flush=True)
            threading.Thread(target=_replace, daemon=True).start()

        return result

    def close_all(self):
        while True:
            try:
                self._pool.get_nowait().close()
            except queue.Empty:
                break

## Lean Setup Helpers

In [ ]:
def _unpack_bin(src: pathlib.Path, dest: pathlib.Path, label: str) -> bool:
    dest.mkdir(parents=True, exist_ok=True)
    print(f'Unpacking {label}: {src} -> {dest} ...')
    t0 = time.time()
    result = subprocess.run(
        ['tar', '-xf', str(src), '-C', str(dest)],
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    if result.returncode != 0:
        print(f'  ERROR unpacking {label}: {result.stderr[:500]}')
    else:
        print(f'  Done in {elapsed:.1f}s')
    return result.returncode == 0


def _patch_lake_git_remotes(project_dir: pathlib.Path) -> None:
    manifest = project_dir / 'lake-manifest.json'
    if not manifest.exists():
        print('  [patch_remotes] lake-manifest.json not found — skipping')
        return
    try:
        data = json.loads(manifest.read_text())
    except Exception as e:
        print(f'  [patch_remotes] failed to parse manifest: {e}')
        return

    packages_dir = project_dir / '.lake' / 'packages'
    subprocess.run(
        ['git', 'config', '--global', '--add', 'safe.directory', '*'],
        capture_output=True, text=True
    )

    for pkg in data.get('packages', []):
        name = pkg.get('name')
        url = pkg.get('url')
        if not name or not url:
            continue
        git_dir = packages_dir / name / '.git'
        if not git_dir.exists():
            print(f'  [patch_remotes] {name}: .git dir missing — skipping')
            continue
        result = subprocess.run(
            ['git', 'remote', 'set-url', 'origin', url],
            cwd=str(packages_dir / name),
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f'  [patch_remotes] {name}: remote set to {url}')
        else:
            print(f'  [patch_remotes] {name}: failed — {result.stderr.strip()[:100]}')


def _setup_lean() -> bool:
    toolchain_bin = DATASET_DIR / 'lean-toolchain.bin'
    project_bin = DATASET_DIR / 'lean-project.bin'
    lean_bin = ELAN_HOME / 'toolchains' / 'leanprover--lean4---v4.29.0-rc6' / 'bin' / 'lean'

    if toolchain_bin.exists() and not lean_bin.exists():
        _unpack_bin(toolchain_bin, ELAN_HOME.parent, 'lean-toolchain')
    elif toolchain_bin.exists() and lean_bin.exists():
        print("Skipping Toolchain_unpack")
    else:
        print(f'WARNING: {toolchain_bin} not found — skipping Lean setup')
        return False

    _lake_dir = LEAN_PROJECT_DIR / '.lake'
    if project_bin.exists() and not _lake_dir.exists():
        _unpack_bin(project_bin, LEAN_PROJECT_DIR.parent, 'lean-project')
    elif project_bin.exists() and _lake_dir.exists():
        print("Skipping LeanProject_unpack")
    else:
        print(f'WARNING: {project_bin} not found — project not unpacked')
        return False

    _patch_lake_git_remotes(LEAN_PROJECT_DIR)
    os.environ['ELAN_HOME'] = str(ELAN_HOME)

    try:
        ver = subprocess.run(
            [str(lean_bin), '--version'],
            capture_output=True, text=True, timeout=5
        )
        if ver.returncode == 0:
            print(f'lean version: {ver.stdout.strip()}')
            return True
        else:
            print(f'lean --version failed: {ver.stderr[:200]}')
            return False
    except FileNotFoundError:
        print('lean binary not found after setup')
        return False
    except subprocess.TimeoutExpired:
        print('lean --version timed out')
        return False

## LeanInterface

In [ ]:
class LeanInterface:
    def __init__(self):
        lean_available = _setup_lean()
        print(f'Lean available: {lean_available}')

        if not LOKAL and REPL_BIN_DATA is not None:
            if not REPL_BIN.exists():
                if REPL_BIN_DATA.exists():
                    REPL_BIN.parent.mkdir(parents=True, exist_ok=True)
                    print(f'Extracting repl binary: {REPL_BIN_DATA} -> {REPL_BIN.parent}')
                    _unpack_bin(REPL_BIN_DATA, REPL_BIN.parent, 'lean-repl')
                    try:
                        REPL_BIN.chmod(0o755)
                    except Exception as exc:
                        print(f'[WARN] chmod repl failed: {exc}', flush=True)
                else:
                    print(f'[WARN] lean-repl.bin nicht gefunden: {REPL_BIN_DATA}', flush=True)
            else:
                print(f'Skipping repl extraction — already exists: {REPL_BIN}')

        self.lean_REPL = LeanREPLPool()
        self.warm_up_test()
        self.lean_repl_warmup()

    def warm_up_test(self):
        _proj = LEAN_PROJECT_DIR
        print('=== Lean project diagnostics ===')
        print(f'  LEAN_PROJECT exists     : {_proj.exists()}')
        print(f'  LEAN_PROJECT contents   : {sorted(p.name for p in _proj.iterdir()) if _proj.exists() else "N/A"}')

        _lake_dir = _proj / '.lake'
        print(f'  .lake/ exists           : {_lake_dir.exists()}')
        if _lake_dir.exists():
            print(f'  .lake/ contents         : {sorted(p.name for p in _lake_dir.iterdir())}')
            _pkg_dir = _lake_dir / 'packages'
            if _pkg_dir.exists():
                print(f'  .lake/packages/         : {sorted(p.name for p in _pkg_dir.iterdir())}')
            else:
                print('  .lake/packages/         : MISSING')

        _manifest = _proj / 'lake-manifest.json'
        if _manifest.exists():
            print(f'  lake-manifest.json      :\n{_manifest.read_text()[:200]}')
        else:
            print('  lake-manifest.json      : NOT FOUND')

        for _lf_name in ['lakefile.lean', 'lakefile.toml']:
            _lf = _proj / _lf_name
            if _lf.exists():
                print(f'  {_lf_name}:\n{_lf.read_text()[:200]}')

        print(f'  ELAN_HOME env           : {os.environ.get("ELAN_HOME", "(not set)")}')
        print(f'  PATH (first 200 chars)  : {os.environ.get("PATH", "")[:200]}')

        _lake_ver = subprocess.run([str(LAKE_BIN), '--version'], capture_output=True, text=True, timeout=10)
        print(f'  lake --version          : {(_lake_ver.stdout + _lake_ver.stderr).strip()[:100]}')

        _mathlib_git_cfg = _lake_dir / 'packages' / 'mathlib' / '.git' / 'config'
        if _mathlib_git_cfg.exists():
            print(f'  mathlib .git/config:\n{_mathlib_git_cfg.read_text()[:200]}')
        else:
            print('  mathlib .git/config : NOT FOUND')
        print(f'  repl binary exists      : {REPL_BIN.exists()} ({REPL_BIN})')
        print('=== End diagnostics ===')

    def lean_repl_warmup(self) -> int:
        """Startet den REPL-Pool und warmt alle Prozesse auf (Mathlib laden).
        Gibt die Anzahl erfolgreich gestarteter Prozesse zurück."""
        ready = self.lean_REPL.lean_repl_warmup(warmup_timeout=WARMUP_TIMEOUT)
        if ready == 0:
            print('[LeanInterface] WARNUNG: Kein REPL-Prozess verfügbar!', flush=True)
        else:
            # Schnell-Check: einen Prozess aus dem Pool nehmen und einfachen Proof testen
            sample = (
                "import Mathlib\nimport Aesop\n\n"
                "set_option maxHeartbeats 0\n\n"
                "open BigOperators Real Nat Topology Rat\n\n"
                "theorem warmup_check : (2 : \u2115) + 2 = 4 := by norm_num\n"
            )
            t0 = time.time()
            cr = self.lean_REPL.check_single(sample, threading.Event(), timeout=15)
            elapsed = round(time.time() - t0, 2)
            if cr.get('complete'):
                print(f'[LeanInterface] REPL-Check OK ({elapsed}s) — pool bereit.', flush=True)
            else:
                print(f'[LeanInterface] REPL-Check FAILED ({elapsed}s): '
                      f'{cr.get("system_errors") or cr.get("errors")}', flush=True)
        return ready

## PythonTool

In [ ]:
class PythonTool:

    def __init__(self, local_jupyter_timeout: float, tool_prompt: str, sandbox=None):

        self._local_jupyter_timeout = local_jupyter_timeout
        self._tool_prompt = tool_prompt
        self._jupyter_session = sandbox

        self._owns_session = sandbox is None

        self._execution_lock = threading.Lock()
        self._init_lock = threading.Lock()

    def _ensure_session(self):

        if self._jupyter_session is None:
            with self._init_lock:
                if self._jupyter_session is None:
                    self._jupyter_session = PythonSandbox(timeout=self._local_jupyter_timeout)

    def _ensure_last_print(self, code: str) -> str:

        lines = code.strip().split('\n')

        if not lines:
            return code

        last_line = lines[-1].strip()

        if re.search(r'\bprint\s*\(', last_line) or last_line.lstrip().startswith(('import ', 'from ')):
            return code

        if not last_line:
            return code

        if last_line.startswith('#'):
            return code

        lines[-1] = 'print(' + last_line + ')'

        return '\n'.join(lines)

    @property
    def instruction(self) -> str:

        return self._tool_prompt

    def process_sync_plus_for_chatml(self, script) -> str:

        self._ensure_session()
        final_script = self._ensure_last_print(script)
        with self._execution_lock:
            try:
                output = self._jupyter_session.execute(final_script)

            except TimeoutError as exc:
                output = f'[ERROR] [TIMEOUT] {exc}'

        return output

## LLMBackend

In [ ]:
class LLMBackend:
    def __init__(self, cfg):
        self.cfg = cfg
        self.port = self.cfg.port
        self.base_url = f'http://0.0.0.0:{self.port}/v1'

        self.client = OpenAI(
            base_url=self.base_url,
            api_key="sk-lokal",
            timeout=self.cfg.session_timeout
        )

        if self.cfg.get_embeddings:
            data = torch.load(self.cfg.embedding_path, map_location="cpu")
            self.embeddings = data["embeddings"]
            print("Loaded embeddings:", self.embeddings.shape)

        self._preload_model_weights()

        self.server_process = self._start_server()

        self._wait_for_server()

    def _preload_model_weights(self) -> None:

        print(f'Loading model weights from {self.cfg.model_path} into OS Page Cache...')
        start_time = time.time()

        files_to_load = []
        total_size = 0

        for root, _, files in os.walk(self.cfg.model_path):
            for file_name in files:
                file_path = os.path.join(root, file_name)

                if os.path.isfile(file_path):
                    files_to_load.append(file_path)
                    total_size += os.path.getsize(file_path)

        def _read_file(path: str) -> None:

            with open(path, 'rb') as file_object:
                while file_object.read(1024 * 1024 * 1024):
                    pass

        with ThreadPoolExecutor(max_workers=self.cfg.workers) as executor:
            list(executor.map(_read_file, files_to_load))

        elapsed = time.time() - start_time
        print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

    def _start_server(self) -> subprocess.Popen:

        cmd = [
            sys.executable,
            '-m',
            'vllm.entrypoints.openai.api_server',
            '--seed',
            str(self.cfg.seed),
            '--model',
            self.cfg.model_path,
            '--served-model-name',
            self.cfg.served_model_name,
            '--tensor-parallel-size',
            '1',
            '--max-num-seqs',
            str(self.cfg.batch_size),
            '--gpu-memory-utilization',
            str(self.cfg.gpu_memory_utilization),
            '--host',
            '0.0.0.0',
            '--port',
            str(self.cfg.port),
            '--dtype',
            self.cfg.dtype,
            '--kv-cache-dtype',
            self.cfg.kv_cache_dtype,
            '--max-model-len',
            str(self.cfg.context_tokens),
        ]

        if hasattr(self.cfg, 'stream_interval'):
            cmd += ['--stream-interval', str(self.cfg.stream_interval)]

        cmd += [
            '--async-scheduling',
            '--disable-log-stats',
            '--enable-prefix-caching',
            "--trust-remote-code",
            "--language-model-only",

        ]

        if self.cfg.quantization != '':
            if self.cfg.load_format_flag!='':
                cmd += ["--load-format", self.cfg.load_format_flag]
            cmd += ["--quantization", self.cfg.quantization]

        if self.cfg.spec_config != {}:
            cmd += ["--speculative-config", json.dumps(self.cfg.spec_config)]

        if self.cfg.compilation_config != {}:
            cmd += ["--compilation-config", json.dumps(self.cfg.compilation_config)]

        if LOKAL:
            cmd += ["--enforce-eager"]
            cmd += ['--max-num-batched-tokens', '4096',]

        self.log_file = open(f'vllm_server_{self.port}.log', 'w')

        return subprocess.Popen(
            cmd,
            stdout=self.log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True
        )

    def _wait_for_server(self):

        print('Waiting for vLLM serverLLM...')
        start_time = time.time()

        for _ in range(self.cfg.server_timeout):
            return_code = self.server_process.poll()

            if return_code is not None:
                self.log_file.flush()

                with open(f'vllm_server_{self.port}.log', 'r') as log_file:
                    logs = log_file.read()

                raise RuntimeError(f'Server died with code {return_code}. Full logs:\n{logs}\n')

            try:
                self.client.models.list()
                elapsed = time.time() - start_time
                print(f'Server is ready (took {elapsed:.2f} seconds).\n')
                return

            except Exception:
                time.sleep(1)

        raise RuntimeError('Server failed to start (timeout).\n')

    def end_server(self) -> None:

        print('Shutting down vLLM server...')

        if self.server_process and self.server_process.poll() is None:
            os.killpg(os.getpgid(self.server_process.pid), signal.SIGTERM)

            try:
                self.server_process.wait(timeout=30)
                print('Server terminated gracefully.')
            except subprocess.TimeoutExpired:
                print('Graceful shutdown timed out, forcing kill...')
                os.killpg(os.getpgid(self.server_process.pid), signal.SIGKILL)
                self.server_process.wait()
                print('Server killed.')

        if hasattr(self, 'log_file') and not self.log_file.closed:
            self.log_file.close()

        self.server_process = None
        print('GPU released.\n')

## Server-Start & predict-Stub

In [ ]:
_init_done = threading.Event()

def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    _init_done.wait()
    return _predict_impl(id_, question, answer)

_inference_server = kaggle_evaluation.aimo_3_inference_server.AIMO3InferenceServer(predict)
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    _inference_server.server.start()

serverLLM = LLMBackend(SolverConfig)

if LEAN_TOOL and not LOKAL:
    serverForm = LLMBackend(FormConfig)
    serverProv = LLMBackend(ProverConfig)

## extract_tool_call

In [ ]:
def extract_tool_call(text: str) -> Optional[dict]:
    """
    Checks if a tool call is present in the text.
    Returns dict with 'name' and 'parameters', or None if no tool call found.

    Supports both XML-style and JSON-style tool calls.
    """

    # --- XML-style: <tool_call><function=name><parameter=x>...</parameter></function></tool_call>
    tool_call_match = re.search(
        r"<tool_call>\s*<function=(\w+)>(.*?)</function>\s*</tool_call>",
        text,
        re.DOTALL
    )
    if tool_call_match:
        name = tool_call_match.group(1).strip()
        inner = tool_call_match.group(2)

        parameters = {}
        for param_match in re.finditer(
                r"<parameter=(\w+)>(.*?)</parameter>",
                inner,
                re.DOTALL
        ):
            param_name = param_match.group(1).strip()
            param_value = param_match.group(2).strip()
            parameters[param_name] = param_value

        return {"name": name, "parameters": parameters}

    # --- JSON-style fallback: {"name": "...", "arguments": {...}}
    json_match = re.search(r"\{.*?\}", text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            name = data.get("name") or data.get("function")
            args = data.get("arguments") or data.get("parameters", {})
            if name and args:
                return {"name": name, "parameters": args}
        except json.JSONDecodeError:
            pass

    return None

## Lean Formalizer

In [ ]:
LEAN4_HEADER = (
    "import Mathlib\n"
    "import Aesop\n\n"
    "set_option maxHeartbeats 0\n\n"
    "open BigOperators Real Nat Topology Rat\n\n"
)

HEADER_MARKER = "open BigOperators Real Nat Topology Rat"
class ProblemEntry(TypedDict):
    name: str
    informal_prefix: str
    formal_statement: str
    lean4_code: str
    problem_id: str
    split: str

class FormalizerClass:
    def __init__(self, cfg_form):
        self.cfg_form = cfg_form
        self.template_form = ChatTemplate()
        self.tokenizer_form = AutoTokenizer.from_pretrained(self.cfg_form.model_path, trust_remote_code=True)
        self.client_form = None  # wird nach Server-Start via set_client_form() gesetzt
        self.messages = []
        self.id_problem = 0

    def set_client_form(self, client_form):
        self.client_form = client_form

    def extract_theorem_part(self,formalizer_output: str) -> str:
        """Entfernt den Header falls vorhanden, gibt nur theorem-Teil zurück."""
        if HEADER_MARKER in formalizer_output:
            return formalizer_output.split(HEADER_MARKER)[-1].strip()
        # Kein Header drin → direkt zurückgeben
        return formalizer_output.strip()

    def build_formal_statement(self,informal_statement_content: str, code: str) -> str:

        theorem_part = self.extract_theorem_part(code)
        informal_prefix = f"/-- {informal_statement_content} -/\n"

        return LEAN4_HEADER + informal_prefix + theorem_part


    def ask_formalizer(self, informal_statement_content:str, theorem_name:str):

        user_prompt = (
        f"Please autoformalize the following natural language problem statement in Lean 4. "
        f"Use the following theorem name: {theorem_name}\n"
        f"The natural language statement is: \n"
        f"{informal_statement_content}"
        f"Think before you provide the lean statement."
        )
        messages = []
        round_form = 0
        while round_form <= self.cfg_form.turns:
            if len(messages) == 0:
                messages = [
                    {"role": "system", "content": self.cfg_form.system_prompt},
                    {"role": "user", "content": user_prompt}
                ]

            tokenizer = self.tokenizer_form
            prompt_ids = self.template_form.apply_chat_template_for_chatml_for_Form(
                messages,
                tokenizer
            )
            max_tokens_available_to_generate = min(
                self.cfg_form.context_tokens - len(prompt_ids),
                self.cfg_form.max_new_tokens,
            )
            if max_tokens_available_to_generate <= 0:
                return {"status": "no_tokens_left", "result": None}

            if max_tokens_available_to_generate < self.cfg_form.buffer_tokens:
                return {"status": "buffer_too_small", "result": None}
            start_time_form_llm = time.time()
            response = self.client_form.completions.create(
                model=self.cfg_form.served_model_name,
                temperature=self.cfg_form.temperature,
                max_tokens=max_tokens_available_to_generate,
                prompt=prompt_ids,
                seed=self.cfg_form.seed,
                extra_body={'top_p': 0.95,}
            )
            deltatime = time.time() - start_time_form_llm
            answer = response.choices[0].text if response.choices else ""
            answer_tokens = self.tokenizer_form.encode(answer)
            print(
                f"FormLLM CALL Token Answer Len {len(answer_tokens)} bei maximal {max_tokens_available_to_generate} in {deltatime}")
            if not answer.strip():
                round_form += 1
                continue

            messages.append({"role": "assistant", "content": answer})

            matches = re.findall(r'```(?:\w+)?\s*(.*?)```', answer, re.DOTALL)

            if matches:
                code = matches[-1].strip()
                informal_prefix = f"/-- {informal_statement_content} -/\n"
                formal_statement = self.build_formal_statement(informal_statement_content, code)
                lean4_code = formal_statement.split(":= by")[0] + ":= by sorry"
                output = ProblemEntry(
                    name=theorem_name,
                    informal_prefix=informal_prefix,
                    formal_statement=formal_statement,
                    lean4_code=lean4_code,
                    split="none",
                    problem_id=str(self.id_problem),
                )
                self.id_problem += 1
                return {"status": "ok", "result": output}

            else:
                # print("Kein Code-Block gefunden")
                round_form +=1
                if round_form > self.cfg_form.turns:
                    return {"status": "turns_exceeded", "result": None}
                else:
                    messages.append({"role": "tool", "content": "No LeanCode was generated."})

if LEAN_TOOL:
    Formalizer = FormalizerClass(FormConfig)
    if not LOKAL:
        Formalizer.set_client_form(serverForm.client)

## Lean Hilfsfunktionen

In [ ]:
def get_error_str(code, errors, error_thres):
    err_str = ""
    code_lines = code.split('\n')
    token_lengths = [len(line) + 1 for line in code_lines]

    error_num_thres = 8 if error_thres else len(errors)

    for i, error in enumerate(errors[:error_num_thres]):
        try:
            start_line = error['pos']['line'] - 1
            start_col = error['pos']['column']

            # Clamp to valid range so out-of-range Lean line numbers don't crash
            start_line = max(0, min(start_line, len(code_lines) - 1))

            if error['endPos'] is None:
                end_line = start_line
                end_col = len(code_lines[start_line])
            else:
                end_line = error['endPos']['line'] - 1
                end_col = error['endPos']['column']

            end_line = max(start_line, min(end_line, len(code_lines) - 1))

            start_char_pos = sum(token_lengths[:start_line]) + start_col
            end_char_pos = sum(token_lengths[:end_line]) + end_col

            err_str += f"\nError {i + 1}:\n"
            err_str += f"\nCorresponding Code:\n```lean4\n"

            error_code = ""
            for ii in range(-4, 0):
                if start_line + ii >= 0:
                    error_code += f"{code_lines[start_line + ii]}\n"
            if start_line != end_line:
                error_code += code_lines[start_line][:start_col] + "<error>" + code_lines[start_line][start_col:] + "\n"

                if not error_thres:
                    for j in range(start_line + 1, end_line):
                        error_code += f"{code_lines[j]}\n"
                else:
                    show_line = 6
                    for j in range(start_line + 1, min(end_line, start_line + show_line)):
                        error_code += f"{code_lines[j]}\n"
                    if end_line > start_line + show_line:
                        leading_spaces = len(code_lines[j]) - len(code_lines[j].lstrip(' '))
                        error_code += "\n" + " " * leading_spaces + "... --[Truncated]-- ...\n"

                error_code += code_lines[end_line][:end_col] + "</error>" + code_lines[end_line][end_col:] + "\n"
            else:
                error_code += code_lines[start_line][:start_col] + "<error>" + code_lines[start_line][
                                                                               start_col:end_col] + "</error>" + code_lines[
                                                                                                                     start_line][
                                                                                                                 end_col:] + "\n"
            if end_line + 1 < len(code_lines):
                error_code += f"{code_lines[end_line + 1]}\n"

            err_str += error_code
            err_str += f"\n```\n"
            err_str += f"\nError Message: {error['data']}\n"
        except (IndexError, KeyError, TypeError):
            err_str += f"\nError {i + 1}: {error.get('data', '[unknown]')} (at pos {error.get('pos')})\n"

    if len(errors) > error_num_thres:
        err_str += f"\n... [Omitted {len(errors) - error_num_thres} more errors] ...\n"

    return err_str


def remove_comments(text):  # remove comments
    # First remove all /- ... -/ blocks
    text = re.sub(r'/-.*?-/', '', text, flags=re.DOTALL)
    # text = re.sub(r'/- (?!special open -/).*?-/', '', text, flags=re.DOTALL)
    # text = re.sub(r'/-{1,2} (?!special open -/).*?-{1,2}/', '', text, flags=re.DOTALL)
    # Then remove -- comments from each line
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        # Split on -- and keep only the first part
        cleaned_line = line.split('--', 1)[0]
        cleaned_lines.append(cleaned_line)
    # Join back together and remove excessive empty lines
    cleaned_text = '\n'.join(cleaned_lines)
    # Remove multiple consecutive empty lines
    # cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)
    return cleaned_text.strip()

def return_theorem_to_prove(text):
    # Pattern that matches from 'theorem' or 'lemma' to ':= by sorry' with any content in between
    pattern = r'((?:theorem).*?:=\s*by\s*sorry)'
    match = re.search(pattern, text, re.DOTALL)
    return match.span() if match else None

def return_theorem_to_replace(text):
    # Pattern that matches from 'theorem' or 'lemma' to ':= by sorry' with any content in between
    # pattern = r'((?:theorem).*?:=\s*by)'
    pattern = r'((?:^|\s)theorem\s+.*?:=\s*by)'
    match = re.search(pattern, text, re.DOTALL)
    return match.span() if match else None

def replace_statement_in_proof(statement, proof):
    if ("apply?" in proof) or ("exact?" in proof):
        return F"**Error**, 'apply?' or 'exact?' is used, which is not allowed."
    stats_re = remove_comments(statement)
    stats_span_ = return_theorem_to_prove(stats_re)
    if stats_span_ is None:
        error_app = '\n'.join(["\n"] + ['-- ' + x for x in statement.split('\n')])
        return F"**Error**, can not find 'theorem' and ':= sorry' in {error_app}"
    proof_str = remove_comments(proof)
    span = return_theorem_to_replace(proof_str)
    if span is None:
        error_app = '\n'.join(["\n"] + ['-- ' + x for x in proof.split('\n')])
        return F"**Error**, can not find 'theorem' and ':=' in {error_app}"
    return stats_re[:stats_span_[1]].replace("sorry", "") + proof_str[span[1]:]

## InferenceHandler & DeepSeekCoTHandler

In [ ]:
class InferenceHandler:
    # Constructor
    def __init__(self):
        pass

    def extrac_code(self, inputs):
        pattern = r'```lean4\n(.*?)\n```'
        matches = re.findall(pattern, inputs, re.DOTALL)
        if matches:
            return matches[-1]
        pattern = r'```lean4\n(.*?)```'
        matches = re.findall(pattern, inputs, re.DOTALL)
        if matches:
            return matches[-1]
        pattern = r'```lean\n(.*?)```'
        matches = re.findall(pattern, inputs, re.DOTALL)
        if matches:
            return matches[-1]
        return "None"

    def clean_code_string(self, code_string):
        # Split the code string into lines
        lines = code_string.splitlines()

        # Filter out lines that start with specified keywords or are blank
        filtered_lines = [
            line for line in lines
            if not (line.startswith("import") or line.startswith("set_option") or line.startswith(
                "open") or line.strip() == "")
        ]

        # Join the remaining lines back into a single string
        cleaned_code = "\n".join(filtered_lines)
        return cleaned_code

    def prover_inference(self, lean4_code, tokenizer):
        pass  # This method must be implemented by any derived class

    def generate_correction_prompt(self, lean4_code_original_stmt,
                                   history_messages_from_prev_round,
                                   prev_round_llm_raw_output,
                                   error_message_for_prev_round,
                                   tokenizer, current_correction_round_num):
        # Returns (prompt_str, messages_list_for_this_prompt)
        raise NotImplementedError

    def problem_check(self, statement, full_code):
        return full_code

class DeepSeekCoTHandler(InferenceHandler):
    def __init__(self):
        pass

    def extrac_code(self, inputs):
        import_head = "import Mathlib\nimport Aesop\n\nset_option maxHeartbeats 0\n\nopen BigOperators Real Nat Topology Rat\n\n"
        pattern = r'```lean4\n(.*?)\n```'
        matches = re.findall(pattern, inputs, re.DOTALL)
        if matches:
            return import_head + matches[-1]
        pattern = r'```lean4\n(.*?)```'
        matches = re.findall(pattern, inputs, re.DOTALL)
        if matches:
            return import_head + matches[-1]
        pattern = r'```lean\n(.*?)```'
        matches = re.findall(pattern, inputs, re.DOTALL)
        if matches:
            return import_head + matches[-1]
        return "None"

    def prover_inference(self, lean4_code, tokenizer):
        formal_statement = lean4_code.split(":= by")[
                               0] + ":= by sorry"  # include sorry https://huggingface.co/deepseek-ai/DeepSeek-Prover-V2-7B
        prompt = F"Complete the following Lean 4 code:\n\n```lean4\n{formal_statement}```\n\nBefore producing the Lean 4 code to formally prove the given theorem, provide a detailed proof plan outlining the main proof steps and strategies.\nThe plan should highlight key ideas, intermediate lemmas, and proof structures that will guide the construction of the final formal proof."
        messages = [
            #{"role": "system", "content": "You are an expert in mathematics and Lean 4."},
            {"role": "user", "content": prompt}
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True
        )
        return text, messages

    def problem_check(self, statement, full_code):
        full_code = replace_statement_in_proof(statement, full_code)
        return full_code

    def generate_correction_prompt(self, lean4_code_original_stmt,
                                   history_messages_from_prev_round,
                                   prev_round_llm_raw_output,
                                   error_message_for_prev_round,
                                   tokenizer, current_correction_round_num):
        original_stmt_for_prompt = lean4_code_original_stmt.split(":= by")[0] + ":= by sorry"

        current_messages = list(history_messages_from_prev_round)

        # Add PREVIOUS assistant's (failed) attempt
        assistant_content = prev_round_llm_raw_output
        current_messages.append({"role": "assistant", "content": assistant_content})

        # Add CURRENT user feedback and request for new attempt
        user_feedback_content = (
            f"The proof (Round {current_correction_round_num - 1}) is not correct. Following is the compilation error message, where we use <error></error> to signal the position of the error.\n\n{error_message_for_prev_round}"
            "\n\nBefore producing the Lean 4 code to formally prove the given theorem, provide a detailed analysis of the error message."
        )
        current_messages.append({"role": "user", "content": user_feedback_content})

        prompt_str = tokenizer.apply_chat_template(current_messages, tokenize=True, add_generation_prompt=True)
        return prompt_str, current_messages

## ProverClass

In [ ]:
class ProverClass:
    def __init__(self, cfg_prov):
        self.cfg_prov = cfg_prov
        self.template_prov = ChatTemplate()
        self.tokenizer_prov = AutoTokenizer.from_pretrained(self.cfg_prov.model_path, trust_remote_code=True)
        self.client_prov = None  # wird nach Server-Start via set_client_prov() gesetzt
        self.LeanInterface = LeanInterface()

    def set_client_prov(self, client_prov):
        self.client_prov = client_prov

    def _call_llm(self, prompt_ids, seed: int | None = None, n: int = 1) -> list[dict]:
        '''Generiert n Varianten in einem einzigen vLLM-Request (n=1 → Einzelausgabe).
        vLLM batcht alle n Completions intern — effizienter als n separate Requests.
        Gibt eine Liste von n Dicts {"status": ..., "result": ...} zurück.'''
        if seed is None:
            seed = self.cfg_prov.seed
        prompts_tokens = len(prompt_ids)

        max_tokens_available_to_generate = min(
            int(self.cfg_prov.context_tokens - prompts_tokens),
            self.cfg_prov.max_new_tokens,
        )
        if max_tokens_available_to_generate <= 0:
            return [{"status": "no_tokens_left", "result": None}] * n

        if max_tokens_available_to_generate < self.cfg_prov.buffer_tokens:
            return [{"status": "buffer_too_small", "result": None}] * n
        start_time_prov_llm = time.time()
        response = self.client_prov.completions.create(
            model=self.cfg_prov.served_model_name,
            temperature=self.cfg_prov.temperature,
            max_tokens=max_tokens_available_to_generate,
            prompt=prompt_ids,
            seed=seed,
            n=n,
            timeout=self.cfg_prov.prover_llm_timeout,
            extra_body={'top_p': 0.95},
        )
        deltatime = time.time()-start_time_prov_llm
        results = []
        for choice in response.choices:
            text = choice.text
            if not text or not text.strip():
                results.append({"status": "empty_generation", "result": ""})
            else:
                answer_tokens = self.tokenizer_prov.encode(text)
                print(
                    f"ProverLLM CALL Token Answer Len {len(answer_tokens)} bei maximal {max_tokens_available_to_generate} für versuch 1/2 in {deltatime}")
                results.append({"status": "ok", "result": text})
        return results


    @staticmethod
    def _score_lean_result(result: dict) -> float:
        """Bewertet ein Lean-Compile-Ergebnis für die Variantenauswahl.
        Höherer Score = besseres Kandidat für die nächste Korrektur-Runde.
        complete → +inf  |  system_error → -inf  |  sonst: Fehleranzahl, Position, Typ."""
        if result.get('complete'):
            return float('inf')
        if result.get('system_errors'):
            return float('-inf')
        errors = result.get('errors', [])
        if not errors:
            return 500.0  # Kein Fehler, aber noch not complete (z.B. sorry)

        score = 0.0
        score -= len(errors) * 20

        # Fehler auf späteren Zeilen = mehr Fortschritt im Beweis
        last_line  = max((e['pos']['line'] for e in errors), default=0)
        first_line = min((e['pos']['line'] for e in errors), default=0)
        score += last_line + first_line * 0.5

        # Fehlertyp-Klassifikation: leichter fixierbare Fehler weniger bestrafen
        for e in errors:
            data = e['data'].lower()
            if 'unsolved goals' in data:
                score -= 2   # häufig, oft mit einem tactic lösbar
            elif 'type mismatch' in data or 'application type mismatch' in data:
                score -= 5
            elif 'unknown identifier' in data or 'unknown tactic' in data:
                score -= 8   # erfordert Import-/Namensfix
            elif 'failed to synthesize' in data:
                score -= 6
            else:
                score -= 3
        return score

    def proof(self, formulizerOutput: ProblemEntry, stop_event: threading.Event, attempt: int) -> dict:
        '''
        Pro Runde werden cfg_prov.variants_per_round LLM-Varianten parallel generiert
        (gleicher Prompt, unterschiedliche Seeds) und parallel mit Lean kompiliert.
        Die per Heuristik beste Variante wird zur Basis der nächsten Korrektur-Runde.

        Abbruch wenn: complete gefunden | deadline überschritten | alle Runden ohne Erfolg.

        Rückgabe:
            {"success": True,  "code": str, "round": int, "errors": None,  "system_errors": None}
            {"success": False, "code": str, "round": int, "errors": list,  "system_errors": str | None}
        '''
        MAX_ROUNDS = self.cfg_prov.ProofCorrection_Rounds
        N_VARIANTS = self.cfg_prov.variants_per_round
        handler = DeepSeekCoTHandler()
        lean_stmt = formulizerOutput["lean4_code"].split(":= by")[0] + ":= by sorry"

        messages_history: list = []
        best_code = ""
        best_errors: list = []
        best_llm_output = ""
        timed_out = False

        deadline = time.time() + self.cfg_prov.proof_timeout

        for round_idx in range(MAX_ROUNDS):
            print(f"--- [Round {round_idx}/{MAX_ROUNDS-1}] ATTEMPT {attempt} ---")

            if stop_event.is_set() or time.time() > deadline:
                timed_out = True
                break

            # Prompt aufbauen — identisch für alle Varianten, Seeds erzeugen Diversität
            if round_idx == 0:
                prompt_str, messages_history = handler.prover_inference(lean_stmt, self.tokenizer_prov)
            else:
                error_str = get_error_str(best_code, best_errors, True)
                prompt_str, messages_history = handler.generate_correction_prompt(
                    lean4_code_original_stmt=lean_stmt,
                    history_messages_from_prev_round=messages_history,
                    prev_round_llm_raw_output=best_llm_output,
                    error_message_for_prev_round=error_str,
                    tokenizer=self.tokenizer_prov,
                    current_correction_round_num=round_idx,
                )

            # N_VARIANTS Varianten in einem einzigen vLLM-Request generieren
            base_seed = self.cfg_prov.seed + attempt * 1000 + round_idx * 10
            llm_outputs = self._call_llm(prompt_str, seed=base_seed, n=N_VARIANTS)
            if not LOKAL:
                if stop_event.is_set() or time.time() > deadline:
                    timed_out = True
                    break

            # Lean-Code aus LLM-Antworten extrahieren
            variants: list[tuple[str, str]] = []  # (llm_raw_output, full_lean_code)
            for llm_out in llm_outputs:
                if llm_out['status'] not in ('ok',):
                    continue
                extracted = handler.extrac_code(llm_out['result'])
                if extracted in (None, 'None'):
                    continue
                full_code = handler.problem_check(lean_stmt, extracted)
                if isinstance(full_code, str) and not full_code.startswith('**Error**'):
                    variants.append((llm_out['result'], full_code))

            if not variants:
                print(f"  Round {round_idx}: Keine verwertbaren Varianten extrahiert — überspringe")
                continue

            # Varianten parallel kompilieren; sobald eine fertig ist, sofort prüfen
            print(f"--- [Round {round_idx}/{MAX_ROUNDS-1}] Kompiliere {len(variants)} Variante(n) ATTEMPT {attempt} ---")
            compile_stop = threading.Event()
            compile_results_map: dict[int, dict] = {}
            complete_result: tuple[str, dict] | None = None

            with ThreadPoolExecutor(max_workers=len(variants)) as compile_exec:
                future_to_idx = {
                    compile_exec.submit(
                        self.LeanInterface.lean_REPL.check_single,
                        code, compile_stop, 15
                    ): i
                    for i, (_, code) in enumerate(variants)
                }
                for future in as_completed(future_to_idx):
                    if stop_event.is_set():
                        compile_stop.set()
                        break
                    idx = future_to_idx[future]
                    try:
                        result = future.result()
                    except Exception as exc:
                        compile_results_map[idx] = {
                            "pass": False, "complete": False,
                            "errors": [], "warnings": [], "system_errors": str(exc)
                        }
                        continue
                    compile_results_map[idx] = result
                    if result.get('complete') and complete_result is None:
                        complete_result = (variants[idx][1], result)
                        compile_stop.set()  # Andere laufende Kompilierung abbrechen
                        break
            compile_stop.set()  # Sicherheitsnetz: beim Verlassen immer setzen

            if complete_result is not None:
                print(f"Proof complete in round {round_idx}!")
                return {"success": True, "code": complete_result[0], "round": round_idx,
                        "errors": None, "system_errors": None}

            if stop_event.is_set() or time.time() > deadline:
                timed_out = True
                break

            # Geordnete Ergebnisliste für Heuristik (fehlende Einträge = abgebrochen)
            compile_results = [
                compile_results_map.get(i, {"pass": False, "complete": False,
                                            "errors": [], "warnings": [], "system_errors": "aborted"})
                for i in range(len(variants))
            ]

            # Beste Variante per Heuristik wählen → Basis für nächste Runde
            scores = [self._score_lean_result(r) for r in compile_results]
            best_i = max(range(len(scores)), key=lambda i: scores[i])
            best_llm_output, best_code = variants[best_i]
            best_result = compile_results[best_i]
            best_errors = best_result.get('errors', [])

            print(f"  Round {round_idx}: Scores={[round(s, 1) for s in scores]}, "
                  f"Variante {best_i} gewählt ({len(best_errors)} Fehler)")

            if best_result.get('system_errors'):
                return {"success": False, "code": best_code, "round": round_idx,
                        "errors": best_errors, "system_errors": best_result['system_errors']}

        sys_err = "prover_timeout" if (timed_out or stop_event.is_set()) else None
        return {"success": False, "code": best_code, "round": MAX_ROUNDS,
                "errors": best_errors, "system_errors": sys_err}


if LEAN_TOOL:
    Prover = ProverClass(ProverConfig)
    if not LOKAL:
        Prover.set_client_prov(serverProv.client)

## SolverClass

In [ ]:
class SolverClass:

    def __init__(self, cfg_llm):

        self.cfg_llm = cfg_llm
        self.template_llm = ChatTemplate()
        self.tokenizer_llm = AutoTokenizer.from_pretrained(self.cfg_llm.model_path, trust_remote_code=True)
        self._tok_vocab = self.tokenizer_llm.get_vocab()  # {token_str: token_id} für O(1)-Lookup
        self.client_llm = None  # wird nach Server-Start gesetzt

        self._initialize_kernels()

        self.notebook_start_time = time.time()
        self.problems_remaining = 50
        self.solved_problems = 0


    def _dummy_calls(self):
        try:
            start_dummy = time.time()
            prompts = [
                "Hello",
                "Explain transformers briefly.",
                "Write a Python function for Fibonacci.",
                "What is machine learning?",
                "Describe GPUs."
            ]

            stream = self.client_llm.completions.create(
                model=self.cfg_llm.served_model_name,
                temperature=0.7,
                logprobs=self.cfg_llm.top_logprobs,
                presence_penalty=0.0,
                max_tokens=64,
                prompt=prompts,
                seed=12345,
                stream=True,
                extra_body={
                    "top_p": 0.95,
                    "return_token_ids": True,
                    "chat_template_kwargs": {"enable_thinking": False}
                }
            )
            num_token = 0
            for chunk in stream:
                new_tokens = chunk.choices[0].token_ids
                num_token += len(new_tokens)
            elapsed = time.time() - start_dummy
            print(f"dummy_calls took {elapsed:.2f} seconds\nthroughput {num_token/elapsed:.2f} tokens/sec\nnumtokens {num_token} tokens\n")
        except Exception as exc:
            print(f"[_dummy_calls] warm-up failed (non-fatal): {exc}", flush=True)

    def _initialize_kernels(self) -> None:

        print(f'Initializing {ATTEMPTS} Jupyter kernels...')
        start_time = time.time()

        self.sandbox_pool = queue.Queue()

        for i in range(ATTEMPTS):
            try:
                self.sandbox_pool.put(
                    PythonSandbox(timeout=self.cfg_llm.jupyter_timeout)
                )
            except Exception as exc:
                print(f"[_initialize_kernels] kernel {i} failed to start (skipping): {exc}", flush=True)

        elapsed = time.time() - start_time
        print(f'Kernels initialized in {elapsed:.2f} seconds\n') # 16 haben 35sec gekostet

    def _scan_for_answer(self, text: str) -> int | None:

        pattern = r'\\boxed\s*\{\s*([0-9,]+)\s*\}'
        matches = re.findall(pattern, text)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value

            except ValueError:
                pass

        pattern = r'final\s+answer\s+is\s*([0-9,]+)'
        matches = re.findall(pattern, text, re.IGNORECASE)

        if matches:
            try:
                clean_value = matches[-1].replace(',', '')
                value = int(clean_value)

                if 0 <= value <= 99999:
                    return value

            except ValueError:
                pass

        return None

    def hoyer_sparsity(self, x: torch.Tensor, dim: int = -1, eps: float = 1e-12):
        """
        Hoyer sparsity for vectors along a dimension.
        Returns value in [0, 1].
        """
        x = x.float()

        l1 = torch.sum(torch.abs(x), dim=dim)
        l2 = torch.sqrt(torch.sum(x ** 2, dim=dim) + eps)

        n = x.shape[dim]
        sqrt_n = torch.sqrt(torch.tensor(n, dtype=x.dtype))
        sparsity = (sqrt_n - (l1 / (l2 + eps))) / (
                    sqrt_n - 1 + eps)

        return sparsity

    def safe_items(self,d):
        if isinstance(d, dict):
            return d.items()
        return []

    def _compute_mean_weighted_entropy(self, logprobs_buffer_list_of_dicts: list) -> float:
        """
        Semantic uncertainty: Hoyer-Sparsity des Embedding-Deltas (token1 vs token2),
        gewichtet mit p(token2). Hohe Sparsity = inhomogene Deltaverteilung = tokens
        inhaltlich sehr verschieden. Vektorisiert über alle Positionen — kein tokenizer.encode.
        """
        if not logprobs_buffer_list_of_dicts:
            return float('inf')

        embeddings = serverLLM.embeddings  # (vocab_size, emb_dim) torch.Tensor
        vocab = self._tok_vocab            # {token_str: token_id}

        chosen_ids = []
        second_ids = []
        p2_values = []

        for top_logprobs_dict in logprobs_buffer_list_of_dicts:
            if not isinstance(top_logprobs_dict, dict) or len(top_logprobs_dict) < 2:
                continue

            # Top-2 nach Logprob
            items = sorted(top_logprobs_dict.items(), key=lambda x: x[1], reverse=True)
            chosen_str, _ = items[0]
            second_str, second_lp = items[1]

            c_id = vocab.get(chosen_str)
            s_id = vocab.get(second_str)
            if c_id is None or s_id is None:
                continue
            if c_id >= len(embeddings) or s_id >= len(embeddings):
                continue

            chosen_ids.append(c_id)
            second_ids.append(s_id)
            p2_values.append(math.exp(second_lp))

        if not chosen_ids:
            return float('inf')

        # Vektorisiert: eine Matrixop für alle Token-Positionen
        c_emb = embeddings[chosen_ids]            # (n, emb_dim)
        s_emb = embeddings[second_ids]            # (n, emb_dim)
        delta = torch.abs(c_emb - s_emb)         # (n, emb_dim)

        eps = 1e-12
        sqrt_n = math.sqrt(delta.shape[1])
        l1 = delta.sum(dim=1)
        l2 = torch.sqrt((delta ** 2).sum(dim=1) + eps)
        sparsity = (sqrt_n - l1 / (l2 + eps)) / (sqrt_n - 1 + eps)  # (n,)

        p2 = torch.tensor(p2_values, dtype=sparsity.dtype, device=sparsity.device)
        return (p2 * sparsity).mean().item()  # höher = semantisch unsicherer

    def search_for_wait_in_top_logprobs(
            self,
            chunkchoicelistitem,
            currently_coding: None | int
    ) -> int:

        if currently_coding == -5:
            return -1

        try:
            top_log_probs_list_of_dicts = chunkchoicelistitem.logprobs.top_logprobs
        except Exception:
            return -1

        if currently_coding is not None:
            top_log_probs_list_of_dicts = top_log_probs_list_of_dicts[:currently_coding]

        if not top_log_probs_list_of_dicts:
            return -1

        for ind, top_logprobs_dict in enumerate(top_log_probs_list_of_dicts):
            if not isinstance(top_logprobs_dict, dict):
                continue
            if len(top_logprobs_dict) == 0:
                continue
            checked_top_logprobs_ind = 0

            for token_str, log_prob in self.safe_items(top_logprobs_dict):
                if checked_top_logprobs_ind >= self.cfg_llm.top_logprobs_search_for_wait:
                    return -1
                if not isinstance(token_str, str):
                    continue
                if token_str.strip() in ("Wait", "WAIT"):
                    return ind - 1
                checked_top_logprobs_ind += 1

        return -1

    def _process_attempt_for_chatml(
            self,
            problem: str,
            system_prompt: str,
            attempt_index: int,
            stop_event: threading.Event,
            deadline: float
    ) -> dict:

        local_tool = None
        sandbox = None

        wait_interceptions = 0
        python_calls = 0
        python_errors = 0
        lean_calls = 0
        lean_positive = 0
        guessed = "False"
        total_python_calc_time = 0.0
        total_tokens = 0
        final_answer = None
        attempt_start = time.time()

        if stop_event.is_set() or time.time() > deadline:
            return {
                    'Attempt': attempt_index + 1,
                    'Guessed': guessed,
                    'Answer': final_answer,
                    'Response Length': total_tokens,
                    "Wait_Interceptions": wait_interceptions,
                    'Python Calls': python_calls,
                    'Python Code Errors': python_errors,
                    'Total Python Calc Time': total_python_calc_time,
                    'Lean Calls': lean_calls,
                    'Positive Lean Proofs': lean_positive,
                    'Weighted Mean Entropy': float('inf'),
            }
        logprobs_buffer = []

        attempt_seed = int(math.pow(self.cfg_llm.seed + attempt_index, 2))

        try:
            sandbox = self.sandbox_pool.get(timeout=self.cfg_llm.sandbox_timeout)

            local_tool = PythonTool(
                local_jupyter_timeout=self.cfg_llm.jupyter_timeout,
                tool_prompt=self.cfg_llm.python_tool_prompt,
                sandbox=sandbox
            )

            tokenizer = self.tokenizer_llm
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": problem}
            ]

            for turn_id in range(self.cfg_llm.turns):
                tokens_before = total_tokens
                currently_coding =  None
                time_left = deadline - time.time()
                print(f"ATTEMPT: {attempt_index} {time_left:.2f} seconds left for this problem")
                if stop_event.is_set() or time.time() > deadline:
                    print("Solver Timeout or stop_event set")
                    break

                prompt_ids = self.template_llm.apply_chat_template_for_chatml_for_Solver(
                    messages,
                    tokenizer
                )
                max_tokens_available_to_generate = min(
                    self.cfg_llm.context_tokens - len(prompt_ids),
                    self.cfg_llm.max_new_tokens,
                )
                if max_tokens_available_to_generate < self.cfg_llm.buffer_tokens:
                    print(f"ATTEMPT: {attempt_index} Available Context window ist kleiner als {self.cfg_llm.buffer_tokens}")
                    break

                enable_thinking = False
                start_gen = time.time()
                stream = self.client_llm.completions.create(
                    model=self.cfg_llm.served_model_name,
                    temperature=self.cfg_llm.temperature,
                    logprobs=self.cfg_llm.top_logprobs,
                    presence_penalty=self.cfg_llm.presence_penalty,
                    max_tokens=max_tokens_available_to_generate,
                    prompt=prompt_ids,
                    seed=attempt_seed,
                    stream=True,
                    extra_body={
                        # 'min_p': self.cfg_llm.min_p,
                        'top_p': 0.95,
                        'return_token_ids': True,
                        'chat_template_kwargs': {'enable_thinking': enable_thinking}
                    }
                )

                try:
                    text_chunks = []

                    for chunk in stream:
                        if stop_event.is_set() or time.time() > deadline:
                            break
                        new_text = ""
                        if chunk.choices[0].token_ids:
                            try:
                                if currently_coding is None:
                                    tag_1 = "<function=python>"
                                    tag_2 = "<function=lean_repl>"
                                    offsets = chunk.choices[0].logprobs.text_offset
                                    chunk_start = offsets[0]

                                    char_index = next((i for i in range(len(chunk.choices[0].text)) if tag_1.startswith(chunk.choices[0].text[i:]) or tag_2.startswith(chunk.choices[0].text[i:])),
                                                      None)
                                    currently_coding = next(
                                        (i for i, offset in enumerate(offsets) if offset - chunk_start >= char_index),
                                        None) if char_index is not None else None
                                else:
                                    currently_coding = -5

                                # Search for Wait in Tokens
                                chunk_logprobs = chunk.choices[0].logprobs

                                found_ind = self.search_for_wait_in_top_logprobs(chunk.choices[0], currently_coding)

                                if found_ind == -1: # No Wait Found
                                    new_tokens = chunk.choices[0].token_ids
                                    new_text = "".join(chunk.choices[0].logprobs.tokens)

                                    total_tokens += len(new_tokens)
                                    text_chunks.append(new_text)

                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs)

                                else:
                                    new_tokens = chunk.choices[0].token_ids[:found_ind]
                                    new_text = "".join(chunk.choices[0].logprobs.tokens[:found_ind])

                                    total_tokens += len(new_tokens)
                                    text_chunks.append(new_text)
                                    text_chunks.append("Wait, I'm not sure. I should think about this again, maybe some tests/experiments or examples with Python, will help me.")

                                    logprobs_buffer.extend(chunk_logprobs.top_logprobs[:found_ind])
                                    wait_interceptions +=1
                                    break
                            except (AttributeError, TypeError, IndexError) as _lp_err:
                                pass  # logprobs absent for this chunk — skip token processing

                        if '}' in new_text:
                            search_text = ''.join(text_chunks[-self.cfg_llm.search_for_answer_tokens:])
                            answer = self._scan_for_answer(search_text)

                            if answer is not None:
                                final_answer = answer
                                break

                finally:
                    print(f"ATTEMPT: {attempt_index} Context ist aktuell zu {float(100*total_tokens/self.cfg_llm.context_tokens):.2f}% voll.")
                    print(f"ATTEMPT: {attempt_index} LLM Call took: {time.time()-start_gen:.2f} seconds with throughput {(total_tokens-tokens_before)/(time.time()-start_gen):.2f} token/sec")
                    print(f"ATTEMPT: {attempt_index} Antwort {"".join(text_chunks)}")
                    stream.close()

                turn_text = "".join(text_chunks)
                # --- Solver turn logging ---
                _log_dir = pathlib.Path('./tmp' if LOKAL else '/kaggle/working')
                _log_dir.mkdir(parents=True, exist_ok=True)
                _turn_num = len([m for m in messages if m['role'] == 'assistant'])
                with open(_log_dir / f'solver_attempt_{attempt_index}.txt', 'a', encoding='utf-8') as _lf:
                    _lf.write(f'--- Turn {_turn_num} | tokens_so_far={total_tokens} ---\n')
                    _lf.write(turn_text)
                    _lf.write('\n\n')
                # ---------------------------
                if final_answer is not None:
                    break
                messages.append({"role": "assistant", "content": turn_text})
                tool = extract_tool_call("".join(text_chunks))
                if tool is not None:
                    if tool["name"] == "python":
                        python_code = tool["parameters"]["code"]
                        python_calls += 1
                        start_pyt = time.time()
                        response_text = local_tool.process_sync_plus_for_chatml(python_code)
                        local_python_calc_time = time.time() - start_pyt
                        print(f"ATTEMPT: {attempt_index} Python Tool took: {local_python_calc_time:.2f} seconds.")
                        if local_python_calc_time<self.cfg_llm.jupyter_timeout:
                            total_python_calc_time += local_python_calc_time

                        if response_text.startswith('[ERROR]') or 'Traceback' in response_text or 'Error:' in response_text:
                            python_errors += 1
                            if response_text.startswith('[ERROR] [TIMEOUT]'):
                                response_text += (
                                    "\n\n[SYSTEM] Python execution exceeded the time limit. "
                                    "Switch to a smarter algorithm: use sympy for symbolic/exact computation, "
                                    "add early-exit conditions, reduce the search space mathematically, "
                                    "or derive a closed-form formula instead of iterating. "
                                    "Only use lean_repl if you need to prove a universal statement "
                                    "(∀ n, ...) that holds for infinitely many cases — not to compute a specific value."
                                )
                        messages.append({"role": "tool", "content": response_text})
                    elif tool["name"] == "lean_repl":
                        print(f"ATTEMPT: {attempt_index} Lean Tool")
                        start_lean = time.time()
                        lean_calls += 1
                        _local_server_form = None
                        try:
                            if LEAN_TOOL and LOKAL:
                                _local_server_form = LLMBackend(FormConfig)
                                Formalizer.set_client_form(_local_server_form.client)

                            formalizer_response_dict = Formalizer.ask_formalizer(tool["parameters"]["informal_statement_content"], tool["parameters"]["theorem_name"])
                        finally:
                            if _local_server_form is not None:
                                _local_server_form.end_server()

                        if formalizer_response_dict["status"] == "no_tokens_left":
                            messages.append({"role": "tool", "content": f"The lean formalizer ran out of context while trying to formalize '{tool['parameters']['theorem_name']}': {tool['parameters']['informal_statement_content']}. Try a simpler or more concise statement."})
                        elif formalizer_response_dict["status"] == "buffer_too_small":
                            messages.append({"role": "tool", "content": f"The lean formalizer ran out of context while trying to formalize '{tool['parameters']['theorem_name']}': {tool['parameters']['informal_statement_content']}. Try a simpler or more concise statement."})
                        elif formalizer_response_dict["status"] == "turns_exceeded":
                            messages.append({"role": "tool", "content": f"The lean formalizer could not produce a valid formal statement for '{tool['parameters']['theorem_name']}' within {Formalizer.cfg_form.turns} attempts. Try rephrasing the statement more precisely."})
                        else:
                            _local_server_prov = None
                            if LEAN_TOOL and LOKAL:
                                _local_server_prov = LLMBackend(ProverConfig)
                                Prover.set_client_prov(_local_server_prov.client)

                            formalizer_response = formalizer_response_dict["result"]
                            stop_event_prover = threading.Event()


                            def run_proof(form, s_event,i):
                                # Prover.proof prüft intern stop_event.is_set() und bricht ab falls True
                                return Prover.proof(form, stop_event=s_event,attempt=i)

                            executor2 = ThreadPoolExecutor(max_workers=self.cfg_llm.profer_cuncurrent)

                            try:
                                futures = []
                                for i in range(self.cfg_llm.profer_cuncurrent):
                                    future = executor2.submit(run_proof, formalizer_response, stop_event_prover,i)
                                    futures.append(future)

                                successful = None
                                results = []

                                for future in as_completed(futures):
                                    try:
                                        result = future.result()
                                        results.append(result)

                                        if result["success"] and successful is None and not stop_event_prover.is_set():
                                            successful = result
                                            stop_event_prover.set()  # alle anderen abbrechen

                                    except Exception as exc:
                                        results.append({"success": False, "code": "", "round": 0, "errors": "", "system_errors": str(exc)})
                                        print(f'Future {exc.__class__.__name__} failed: {exc}')
                                        continue
                            finally:
                                stop_event_prover.set()
                                executor2.shutdown(wait=True, cancel_futures=True)
                                if _local_server_prov is not None:
                                    _local_server_prov.end_server()
                            print(f"ATTEMPT: {attempt_index} Lean Tool took: {time.time() - start_lean:.2f} seconds.")

                            if successful:
                                lean_positive += 1
                                messages.append({"role": "tool",
                                                 "content": (
                                                     f"LEAN4 PROOF SUCCESSFUL: '{tool['parameters']['informal_statement_content']}' "
                                                     f"has been formally verified as TRUE. "
                                                     f"You can rely on this result with certainty."
                                                 )})
                            else:
                                clean_failures = [r for r in results if r["system_errors"] is None]
                                timeouts = [r for r in results if r["system_errors"] == "prover_timeout"]
                                system_errors = [r["system_errors"] for r in results
                                                 if r["system_errors"] is not None
                                                 and r["system_errors"] != "prover_timeout"]

                                if clean_failures:
                                    messages.append({"role": "tool",
                                                     "content": f"The theorem {tool['parameters']['theorem_name']} failed "
                                                                f"in {len(clean_failures)}/{self.cfg_llm.profer_cuncurrent} attempts with a clean failure. "
                                                                f"Most likely it's not correct. Don't ask again to prove it."})

                                elif system_errors or timeouts:
                                    messages.append({"role": "tool",
                                                     "content": f"The theorem {tool['parameters']['theorem_name']} could not "
                                                                f"be validated due to technical issues ({len(timeouts)} timeouts, "
                                                                f"{len(system_errors)} system errors) in a total of {self.cfg_llm.profer_cuncurrent} proof attempts. This result is neutral. "
                                                                f"Don't ask again to prove it."})
                else:
                    answer_text = messages[-1]["content"]
                    final_answer = self._scan_for_answer(answer_text)
                    if final_answer is not None:
                        break

            if final_answer is None:
                # Guess: pre-fill "\boxed{" as partial assistant message so the model
                # can only continue with the number — no explanations possible.
                messages[0]["content"] = messages[0]["content"][:messages[0]["content"].find("answer")] +"\nOutput Format:The final answer must be a non-negative integer between 0 and 99999.Place your final numerical answer inside \\boxed{}, e.g., \\boxed{42}"
                messages[1]["content"] = messages[1]["content"][:messages[1]["content"].find("CORRECT USE")]
                messages.append({"role": "user", "content": "Based on your work above, what is your best integer answer? No discussions! No Evaluation! No Python! Just print your best integer guess!"}) #It's the most important thing you can do now! "})


                prompt_ids = self.template_llm.apply_chat_template_for_chatml_for_Solver_final(
                    messages,
                    tokenizer
                )

                stream = self.client_llm.completions.create(
                    model=self.cfg_llm.served_model_name,
                    temperature=0.0,
                    logprobs=self.cfg_llm.top_logprobs,
                    max_tokens=self.cfg_llm.guessing_tokens,
                    prompt=prompt_ids,
                    seed=attempt_seed,
                    stream=True,
                    extra_body={
                        'top_p': 0.95,
                        'return_token_ids': True,
                        'chat_template_kwargs': {'enable_thinking': False}
                    }
                )
                try:
                    text_chunks = []

                    for chunk in stream:

                        if chunk.choices[0].token_ids:

                            chunk_logprobs = chunk.choices[0].logprobs
                            new_tokens = chunk.choices[0].token_ids
                            total_tokens += len(new_tokens)

                            new_text = "".join(chunk.choices[0].logprobs.tokens)
                            text_chunks.append(new_text)
                            logprobs_buffer.extend(chunk_logprobs.top_logprobs)

                finally:
                    stream.close()
                # Prefix was pre-filled into prompt — prepend it for answer scanning
                generated = "".join(text_chunks)
                final_answer = self._scan_for_answer(generated)
                if final_answer is None:
                    guessed = "Tried"
                else:
                    guessed = "True"

        except Exception as exc:
            print(f'[Solver] Unerwartete Exception in Attempt {attempt_index}: {exc.__class__.__name__}: {exc}', flush=True)
            raise

        finally:
            if sandbox is not None:
                try:
                    sandbox.reset()
                except Exception:
                    try:
                        sandbox.close()
                    except Exception:
                        pass
                    try:
                        sandbox = PythonSandbox(timeout=self.cfg_llm.jupyter_timeout)
                    except Exception:
                        sandbox = None
                if sandbox is not None:
                    self.sandbox_pool.put(sandbox)

        mean_weighted_entropy = self._compute_mean_weighted_entropy(logprobs_buffer)
        time_attempt = time.time()-attempt_start
        print(f"ATTEMPT: {attempt_index} finished in {time_attempt:.2f} sec: Guessed? {guessed}")

        return {
            'Attempt': attempt_index + 1,
            'Guessed': guessed,
            'Response Length': total_tokens,
            "Wait_Interceptions": wait_interceptions,
            'Python Calls': python_calls,
            'Python Code Errors': python_errors,
            'Total Python Calc Time': total_python_calc_time,
            'Lean Calls': lean_calls,
            'Positive Lean Proofs': lean_positive,
            'Weighted Mean Entropy': mean_weighted_entropy,
            'Answer': final_answer,

        }

    def _select_answer(self, detailed_results: list) -> int:

        answer_weights = defaultdict(float)
        answer_votes = defaultdict(int)
        answer_python_calc_time = defaultdict(float)
        for result in detailed_results:
            answer = result['Answer']
            if answer is not None:

                entropy = result['Weighted Mean Entropy']
                if result['Guessed'] == "True":
                    entropy = entropy * 1.4
                elif result['Guessed'] == "False":
                    entropy = entropy * 1


                weight = 1.0 / max(entropy, 1e-9)

                answer_python_calc_time[answer] += result['Total Python Calc Time']
                answer_weights[answer] += weight
                answer_votes[answer] += 1

        sorted_answers_python_calc_time = sorted(
            answer_python_calc_time,
            key=answer_python_calc_time.get,
            reverse=True
        )
        python_calc_factors = [1.1,1.075,1.05,1.025,1.0,1.0,1.0,1.0]
        for rank, answer in enumerate(sorted_answers_python_calc_time):
            if rank < len(python_calc_factors):
                answer_weights[answer] *= python_calc_factors[rank]

        scored_answers = []

        for answer, total_weight in answer_weights.items():
            scored_answers.append({
                'answer': answer,
                'votes': answer_votes[answer],
                'score': total_weight
            })

        if not scored_answers:
            print('\nFinal Answer: 0\n')
            return 0

        scored_answers.sort(key=lambda x: x['score'], reverse=True)

        vote_data = []

        for item in scored_answers:
            vote_data.append((
                item['answer'],
                item['votes'],
                item['score']
            ))

        vote_dataframe = pd.DataFrame(
            vote_data,
            columns=['Answer', 'Votes', 'Score']
        )

        vote_dataframe = vote_dataframe.round({'Score': 5})
        print(vote_dataframe)

        final_answer = scored_answers[0]['answer']
        print(f'\nFinal Answer: {final_answer}\n')

        return final_answer

    def solve_problem(self, problem: str) -> int:

        print(f'\nProblem: {problem}\n')

        user_input = f'{problem} {self.cfg_llm.preference_python_prompt}'
        elapsed_global = time.time() - self.notebook_start_time
        time_left = self.cfg_llm.notebook_limit - elapsed_global
        problems_left_others = max(0, self.problems_remaining - 1)
        reserved_time = problems_left_others * self.cfg_llm.base_problem_timeout
        budget = time_left - reserved_time
        budget = min(budget, self.cfg_llm.high_problem_timeout)
        budget = max(budget, self.cfg_llm.base_problem_timeout)

        deadline = time.time() + budget

        print(f'Budget: {budget:.2f} seconds | Deadline: {deadline:.2f}\n')

        tasks = []

        for attempt_index in range(ATTEMPTS):
            tasks.append((self.cfg_llm.system_prompt, attempt_index))

        detailed_results = []
        valid_answers = []

        stop_event = threading.Event()

        executor = ThreadPoolExecutor(max_workers=self.cfg_llm.workers)

        try:
            futures = []

            for (system_prompt, attempt_index) in tasks:
                future = executor.submit(
                    self._process_attempt_for_chatml,
                    user_input,
                    system_prompt,
                    attempt_index,
                    stop_event,
                    deadline
                )

                futures.append(future)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    detailed_results.append(result)

                    if result['Answer'] is not None:
                        valid_answers.append(result['Answer'])

                    counts = Counter(valid_answers).most_common(1)

                    if counts and counts[0][1] >= self.cfg_llm.early_stop:
                        stop_event.set()

                        for f in futures:
                            f.cancel()

                        break

                except Exception as exc:
                    print(f'Future failed: {exc}')
                    continue

        finally:
            stop_event.set()
            executor.shutdown(wait=True, cancel_futures=True)

            self.problems_remaining = max(0, self.problems_remaining - 1)

        if detailed_results:
            results_dataframe = pd.DataFrame(detailed_results)
            results_dataframe['Weighted Mean Entropy'] = results_dataframe['Weighted Mean Entropy'].round(3)
            results_dataframe['Answer'] = results_dataframe['Answer'].astype('Int64')

            print(results_dataframe)

        if not valid_answers:
            print('\nResult: 0\n')

            return 0

        return self._select_answer(detailed_results)

    def __del__(self):

        if hasattr(self, 'sandbox_pool'):
            while not self.sandbox_pool.empty():
                try:
                    sb = self.sandbox_pool.get_nowait()
                    sb.close()

                except Exception:
                    pass

## Solver-Instanz & _predict_impl

In [ ]:
solver = SolverClass(SolverConfig)
solver.client_llm = serverLLM.client

def _predict_impl(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    try:
        id_value = id_.item(0)
        question_text = question.item(0)
    except Exception as exc:
        print(f"[predict] failed to extract input: {exc}", flush=True)
        return pl.DataFrame({'id': 0, 'answer': 0})

    gc.disable()
    try:
        final_answer = solver.solve_problem(question_text)
    except Exception as exc:
        print(f"[predict] solve_problem raised unexpectedly: {exc}", flush=True)
        final_answer = 0
    finally:
        gc.enable()
        gc.collect()

    return pl.DataFrame({'id': id_value, 'answer': final_answer})

# Initialisierung abgeschlossen — predict() darf jetzt Requests bearbeiten
_init_done.set()

## Main

In [ ]:
df = pd.read_csv(
    "./kaggle_evaluation/reference.csv"
).drop("answer", axis=1, errors="ignore").to_csv("reference.csv", index=False)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    _inference_server.server.wait_for_termination()   # blockiert bis alle Requests fertig

else:
    # _inference_server.run_local_gateway(('./data/reference-onehardest.csv',))
    if LOKAL:
        _inference_server.run_local_gateway(('./kaggle_evaluation/reference.csv',))
    else:
        _inference_server.run_local_gateway(('/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/reference.csv',))


total_sec = time.time() - overall_start_time
print(total_sec)